In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags

df = load_all_snapshots()
f = add_discipline_flags(df)

# Target population: swings only. P(Whiff | Swing).
swings = f[f["is_swing"]].copy()
swings["target"] = swings["is_whiff"].astype(int)

print(f"swings: {len(swings):,}")
print(f"whiff rate: {swings['target'].mean():.4f}")
print()
print(swings["game_date"].min(), "->", swings["game_date"].max())

swings: 338,364
whiff rate: 0.2322

2024-03-28 -> 2024-09-29


In [2]:
from src.utils.temporal import split_by_date

split = split_by_date(
    swings,
    train_end="2024-07-14",      # through the day before the All-Star break
    validation_end="2024-08-31",
)

for name, part in [("train", split.train),
                   ("validation", split.validation),
                   ("test", split.test)]:
    print(f"{name:11s} {len(part):7,}  whiff {part['target'].mean():.4f}")
print()
print(split.summary())

train       200,735  whiff 0.2301
validation   83,492  whiff 0.2312
test         54,137  whiff 0.2413

{'train': {'rows': 200735, 'start': '2024-03-28', 'end': '2024-07-14'}, 'validation': {'rows': 83492, 'start': '2024-07-19', 'end': '2024-08-31'}, 'test': {'rows': 54137, 'start': '2024-09-01', 'end': '2024-09-29'}}


In [3]:
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

league_rate = split.train["target"].mean()
print(f"train whiff rate: {league_rate:.4f}")

def evaluate(y_true, y_pred, name):
    return {
        "model": name,
        "log_loss": log_loss(y_true, y_pred),
        "brier": brier_score_loss(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_pred) if len(np.unique(y_pred)) > 1 else np.nan,
    }

results = []
for part_name, part in [("validation", split.validation), ("test", split.test)]:
    pred = np.full(len(part), league_rate)
    r = evaluate(part["target"], pred, "constant")
    r["split"] = part_name
    results.append(r)

print(pd.DataFrame(results).round(5).to_string(index=False))

train whiff rate: 0.2301
   model  log_loss   brier  auc      split
constant   0.54072 0.17775  NaN validation
constant   0.55294 0.18321  NaN       test


In [4]:
LOOKUP_KEYS = ["pitch_type", "balls", "strikes", "stand", "p_throws"]

def fit_lookup(train, keys, prior_strength=50.0):
    """Grouped mean with shrinkage toward the global rate.

    Shrinkage matters: some cells have very few swings, and an unshrunk
    cell mean of 0.0 or 1.0 produces infinite log loss.
    """
    global_rate = train["target"].mean()
    tbl = train.groupby(keys)["target"].agg(["sum", "size"])
    tbl["rate"] = (tbl["sum"] + global_rate * prior_strength) / (tbl["size"] + prior_strength)
    return tbl["rate"], global_rate

def predict_lookup(part, table, fallback, keys):
    idx = pd.MultiIndex.from_frame(part[keys])
    return table.reindex(idx).fillna(fallback).to_numpy()

lookup_table, global_rate = fit_lookup(split.train, LOOKUP_KEYS)
print(f"lookup cells: {len(lookup_table)}")
print(lookup_table.describe().round(4).to_string())

for part_name, part in [("validation", split.validation), ("test", split.test)]:
    pred = predict_lookup(part, lookup_table, global_rate, LOOKUP_KEYS)
    r = evaluate(part["target"], pred, "lookup")
    r["split"] = part_name
    results.append(r)

print()
print(pd.DataFrame(results).round(5).to_string(index=False))

lookup cells: 583
count    583.0000
mean       0.2422
std        0.0577
min        0.0911
25%        0.2155
50%        0.2366
75%        0.2786
max        0.4080

   model  log_loss   brier     auc      split
constant   0.54072 0.17775     NaN validation
constant   0.55294 0.18321     NaN       test
  lookup   0.52349 0.17176 0.62551 validation
  lookup   0.53544 0.17697 0.62546       test


In [5]:
from src.features.strike_zone import in_strike_zone

for part in [split.train, split.validation, split.test]:
    part["in_zone_flag"] = in_strike_zone(part)

LOOKUP2_KEYS = ["pitch_type", "balls", "strikes", "in_zone_flag"]

lookup2, global2 = fit_lookup(split.train, LOOKUP2_KEYS)
print(f"cells: {len(lookup2)}")

for part_name, part in [("validation", split.validation), ("test", split.test)]:
    pred = predict_lookup(part, lookup2, global2, LOOKUP2_KEYS)
    r = evaluate(part["target"], pred, "lookup+zone")
    r["split"] = part_name
    results.append(r)

res = pd.DataFrame(results)
print()
print(res[res["split"] == "validation"].round(5).to_string(index=False))

cells: 329

      model  log_loss   brier     auc      split
   constant   0.54072 0.17775     NaN validation
     lookup   0.52349 0.17176 0.62551 validation
lookup+zone   0.48524 0.15629 0.70892 validation


In [6]:
import json

PROJECT = Path.cwd().parents[1]
out = PROJECT / "models" / "artifacts" / "whiff_baseline"
out.mkdir(parents=True, exist_ok=True)

res.to_csv(out / "baseline_metrics.csv", index=False)

meta = {
    "target": "P(whiff | swing)",
    "train": split.summary()["train"],
    "validation": split.summary()["validation"],
    "test": split.summary()["test"],
    "league_rate_train": float(league_rate),
    "lookup_keys": LOOKUP_KEYS,
    "lookup2_keys": LOOKUP2_KEYS,
    "prior_strength": 50.0,
}
(out / "split_metadata.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

{
  "target": "P(whiff | swing)",
  "train": {
    "rows": 200735,
    "start": "2024-03-28",
    "end": "2024-07-14"
  },
  "validation": {
    "rows": 83492,
    "start": "2024-07-19",
    "end": "2024-08-31"
  },
  "test": {
    "rows": 54137,
    "start": "2024-09-01",
    "end": "2024-09-29"
  },
  "league_rate_train": 0.2301043664532842,
  "lookup_keys": [
    "pitch_type",
    "balls",
    "strikes",
    "stand",
    "p_throws"
  ],
  "lookup2_keys": [
    "pitch_type",
    "balls",
    "strikes",
    "in_zone_flag"
  ],
  "prior_strength": 50.0
}


In [7]:
val = split.validation.copy()
val["pred"] = predict_lookup(val, lookup2, global2, LOOKUP2_KEYS)

# 예측 구간별 실제 whiff율 — 캘리브레이션 미리보기
val["bucket"] = pd.cut(val["pred"], bins=[0, .10, .15, .20, .25, .30, .40, 1.0])
cal = val.groupby("bucket", observed=True).agg(
    n=("target", "size"),
    predicted=("pred", "mean"),
    actual=("target", "mean"),
)
cal["gap"] = cal["actual"] - cal["predicted"]
print(cal.round(4).to_string())

                 n  predicted  actual     gap
bucket                                       
(0.0, 0.1]    9016     0.0885  0.0821 -0.0064
(0.1, 0.15]  12704     0.1300  0.1293 -0.0007
(0.15, 0.2]  30798     0.1704  0.1647 -0.0057
(0.2, 0.25]   7483     0.2228  0.2198 -0.0029
(0.25, 0.3]   5120     0.2777  0.2896  0.0120
(0.3, 0.4]    6745     0.3418  0.3647  0.0229
(0.4, 1.0]   11626     0.5151  0.5384  0.0233


In [8]:
# 아직 안 쓴 정보가 얼마나 있나
unused = ["release_speed", "pfx_x", "pfx_z", "plate_x", "plate_z",
          "release_spin_rate", "release_extension"]
print()
print("correlation with whiff among swings:")
for c in unused:
    v = pd.to_numeric(val[c], errors="coerce")
    print(f"  {c:20s} {v.corr(val['target']):+.4f}")


correlation with whiff among swings:
  release_speed        -0.1025
  pfx_x                +0.0452
  pfx_z                -0.1007
  plate_x              +0.0772
  plate_z              -0.1770
  release_spin_rate    +0.0147
  release_extension    +0.0118


In [9]:
print("within pitch type:")
for pt, g in val.groupby("pitch_type"):
    if len(g) < 2000:
        continue
    spin = pd.to_numeric(g["release_spin_rate"], errors="coerce")
    velo = pd.to_numeric(g["release_speed"], errors="coerce")
    pz = pd.to_numeric(g["plate_z"], errors="coerce")
    print(f"  {pt}: spin {spin.corr(g['target']):+.3f}  "
          f"velo {velo.corr(g['target']):+.3f}  "
          f"plate_z {pz.corr(g['target']):+.3f}  (n={len(g):,})")

within pitch type:
  CH: spin +0.015  velo +0.004  plate_z -0.299  (n=9,157)
  CU: spin +0.042  velo +0.053  plate_z -0.468  (n=4,615)
  FC: spin +0.024  velo +0.003  plate_z -0.093  (n=6,883)
  FF: spin +0.051  velo +0.070  plate_z +0.241  (n=27,700)
  FS: spin -0.099  velo -0.072  plate_z -0.367  (n=2,915)
  SI: spin +0.015  velo +0.037  plate_z +0.011  (n=12,142)
  SL: spin +0.039  velo +0.027  plate_z -0.384  (n=12,168)
  ST: spin +0.044  velo +0.050  plate_z -0.323  (n=5,737)
